# Обзор данных

Что выгружено из ISS API, сколько получилось событий и какие из них пригодны
для анализа. Ноутбук запускается после `make data && make splits`.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from src import moex, pipeline
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
P = moex.DATA_PROCESSED

## Состав индекса

Ключевая деталь: эндпоинт отдаёт по одному интервалу членства на бумагу,
а не полную историю. И `till` у действующих членов — это край данных,
а не дата исключения.

In [2]:
composition = pd.read_parquet(moex.DATA_RAW / 'index_composition.parquet')
print(f'записей: {len(composition)}, уникальных тикеров: {composition.ticker.nunique()}')

last_snapshot = composition.date_till.max()
print(f'последний срез состава: {last_snapshot.date()}')
print(f'бумаг в индексе на эту дату: {(composition.date_till >= last_snapshot).sum()}')
composition.head()

записей: 127, уникальных тикеров: 127
последний срез состава: 2026-09-18
бумаг в индексе на эту дату: 44


,ticker,date_from,date_till,tradingsession,index_id
0,AFKS,2012-09-18,2026-09-18,3,IMOEX
1,AFLT,2001-04-28,2026-09-18,3,IMOEX
2,AGRO,2015-12-16,2024-12-02,3,IMOEX
3,AKRN,2012-12-18,2017-12-21,3,IMOEX
4,ALRS,2012-12-18,2026-09-18,3,IMOEX


## Торговый календарь и разрыв 2022 года

In [3]:
index_prices = pd.read_parquet(moex.DATA_RAW / 'index_prices.parquet').sort_values('TRADEDATE')
gaps = index_prices.TRADEDATE.diff().dt.days
for position in gaps[gaps > 7].index:
    print(f'перерыв {int(gaps[position])} дней: '
          f'{index_prices.TRADEDATE[position - 1].date()} -> {index_prices.TRADEDATE[position].date()}')

перерыв 27 дней: 2022-02-25 -> 2022-03-24


В дни остановки торгов ISS отдаёт строки с пустой ценой. Считать их
торговыми нельзя, поэтому календарь берётся по индексу, а дни бумаги — по
наличию цены закрытия.

In [4]:
quotes = pd.read_parquet(moex.DATA_RAW / 'quotes_events.parquet')
halt = quotes[quotes.TRADEDATE.between('2022-02-28', '2022-03-23')]
print(f'строк за период остановки: {len(halt)}')
print(f'из них с ценой закрытия: {halt.CLOSE.notna().sum()}')

строк за период остановки: 416
из них с ценой закрытия: 0


## События и их пригодность

In [5]:
data = pipeline.load()
events = data.events
print(f'всего событий: {len(events)}')
print(events.event_type.value_counts().to_string())
print()
print(f'для анализа цены: {events.in_price_sample.sum()}')
print(f'для анализа ликвидности: {events.in_liquidity_sample.sum()}')

всего событий: 42
event_type
исключение    23
включение     19

для анализа цены: 26
для анализа ликвидности: 29


In [6]:
columns = ['ticker', 'event_type', 'event_date', 'n_estimation',
           'last_quote_t', 'delisted', 'in_price_sample', 'drop_reason']
view = events[columns].copy()
view['event_date'] = view.event_date.dt.date
view

,ticker,event_type,event_date,n_estimation,last_quote_t,delisted,in_price_sample,drop_reason
0,POGR,исключение,2022-06-30,207,8,True,False,бумага ушла с торгов: событие смешано с корпор...
1,HHRU,исключение,2022-09-16,207,482,True,False,бумага ушла с торгов: событие смешано с корпор...
2,DSKY,исключение,2023-03-17,211,261,True,False,бумага ушла с торгов: событие смешано с корпор...
3,SGZH,включение,2023-06-16,211,831,False,True,
4,FIXP,исключение,2023-09-22,211,443,True,False,бумага ушла с торгов: событие смешано с корпор...
5,POSI,включение,2023-09-22,211,761,False,True,
6,SELG,включение,2023-09-22,211,761,False,True,
7,FLOT,включение,2023-12-22,211,696,False,True,
8,SMLT,включение,2023-12-22,211,696,False,True,
9,QIWI,исключение,2024-02-27,211,440,True,False,бумага ушла с торгов: событие смешано с корпор...


## Почему события приходят пачками

Индекс пересматривается раз в квартал, поэтому события кластеризованы по
датам. Это определяет весь дальнейший статистический вывод: наблюдения
внутри одной даты зависимы.

In [7]:
clusters = events.groupby('event_date').agg(
    n=('ticker', 'size'), tickers=('ticker', lambda s: ' '.join(s)))
print(f'событий: {len(events)}, дат: {len(clusters)}, максимум в одну дату: {clusters.n.max()}')
clusters

событий: 42, дат: 21, максимум в одну дату: 6


,n,tickers
event_date,,
2022-06-30,1,POGR
2022-09-16,1,HHRU
2023-03-17,1,DSKY
2023-06-16,1,SGZH
2023-09-22,3,FIXP POSI SELG
2023-12-22,2,FLOT SMLT
2024-02-27,1,QIWI
2024-03-22,2,POLY YNDX
2024-06-21,1,LEAS


## Пары редомициляции

In [8]:
from src import sample
sample.validate_redomicile_pairs(data.quotes)

,old,new,last_old,first_new,gap_days,consistent,note
0,TCSG,T,2024-11-27,2024-11-28,1,True,новый тикер начал торговаться после ухода старого
1,YNDX,YDEX,2024-06-14,2024-07-24,40,True,новый тикер начал торговаться после ухода старого
2,FIVE,X5,2024-04-03,2025-01-09,281,True,новый тикер начал торговаться после ухода старого
3,HHRU,HEAD,2024-08-09,2024-09-26,48,True,новый тикер начал торговаться после ухода старого
4,AGRO,RAGR,2024-12-02,2025-02-17,77,True,новый тикер начал торговаться после ухода старого
